# XLM-RoBERTa-Large Inference on AWS Trainium2

This notebook demonstrates how to run `FacebookAI/xlm-roberta-large` (559M parameters, 24 layers, 100+ languages) on AWS Trainium2 using `torch_neuronx`.

**What you'll learn:**
1. Trace and compile an encoder-only transformer for Neuron
2. Validate numerical accuracy (Neuron vs CPU)
3. Scale inference with DataParallel across NeuronCores
4. Compare throughput with GPU baselines
5. Explore NKI custom kernel development

**Instance requirements:** trn2.3xlarge (1 Trainium2 chip, 4 NeuronCores at LNC=2)

**SDK version:** Neuron SDK 2.29 (torch-neuronx 2.9.0)

## 1. Setup

Activate the pre-installed PyTorch inference environment and import dependencies.

In [1]:
import os
import time
import torch
import torch.nn.functional as F
import torch_neuronx
from transformers import XLMRobertaTokenizer, XLMRobertaModel

# Environment info
print(f"PyTorch version: {torch.__version__}")
print(f"torch-neuronx version: {torch_neuronx.__version__}")
print(f"Neuron cores available: {torch_neuronx.xla_impl.data_parallel.device_count()}")

PyTorch version: 2.9.1+cu128
torch-neuronx version: 2.9.0.2.13.24727+8e870898


Neuron cores available: 4


## 2. Load Model and Tokenizer

XLM-RoBERTa-Large is a multilingual encoder-only transformer:
- **559M parameters** (24 layers, 16 heads, head_dim=64, hidden=1024)
- **250K vocabulary** (SentencePiece, 100+ languages)
- Same architecture as RoBERTa-Large, trained on 2.5TB of CommonCrawl data in 100 languages

In [2]:
tokenizer = XLMRobertaTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")
model = XLMRobertaModel.from_pretrained("FacebookAI/xlm-roberta-large")
model.config.return_dict = False  # Required for torch_neuronx.trace()
model.eval()

param_count = sum(p.numel() for p in model.parameters())
param_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**2)
print(f"Parameters: {param_count:,} ({param_mb:.0f} MB in FP32)")

huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parameters: 559,890,432 (2136 MB in FP32)


## 3. Multilingual Inference on CPU

Before compiling for Neuron, let's verify the model works with multilingual text.

In [3]:
# Multilingual examples demonstrating XLM-RoBERTa's capabilities
texts = {
    "English": "Machine learning models can understand multiple languages simultaneously.",
    "French": "Les modeles d'apprentissage automatique peuvent comprendre plusieurs langues.",
    "German": "Maschinelle Lernmodelle koennen mehrere Sprachen gleichzeitig verstehen.",
    "Chinese": "机器学习模型可以同时理解多种语言。",
    "Arabic": "يمكن لنماذج التعلم الآلي فهم لغات متعددة في وقت واحد.",
    "Japanese": "機械学習モデルは複数の言語を同時に理解できます。",
}

print("CPU inference (multilingual embeddings):")
print("-" * 60)
for lang, text in texts.items():
    inputs = tokenizer(text, return_tensors="pt", padding="max_length",
                       max_length=128, truncation=True)
    with torch.no_grad():
        outputs = model(inputs['input_ids'], inputs['attention_mask'])
    # outputs is a tuple; first element is last_hidden_state
    cls_embedding = outputs[0][:, 0, :]
    print(f"  {lang:>8}: shape={cls_embedding.shape}, norm={cls_embedding.norm():.4f}")

CPU inference (multilingual embeddings):
------------------------------------------------------------
   English: shape=torch.Size([1, 1024]), norm=31.1807


    French: shape=torch.Size([1, 1024]), norm=31.1791
    German: shape=torch.Size([1, 1024]), norm=31.1821


   Chinese: shape=torch.Size([1, 1024]), norm=31.1762
    Arabic: shape=torch.Size([1, 1024]), norm=31.1823


  Japanese: shape=torch.Size([1, 1024]), norm=31.1787


## 4. Trace and Compile for Neuron

We compile the model using `torch_neuronx.trace()` with optimized compiler arguments:
- `--auto-cast matmult`: Cast matrix multiplications to BF16 for **2.5x throughput** with negligible accuracy loss
- `--optlevel 2`: Aggressive compiler optimizations
- `--model-type transformer`: Enable transformer-specific optimizations

In [4]:
# Create example inputs for tracing (seq_len=128)
SEQ_LEN = 128
example_inputs = tokenizer(
    "This is an example sentence for tracing.",
    return_tensors="pt",
    padding="max_length",
    max_length=SEQ_LEN,
    truncation=True
)

# Compiler args: matmult autocast gives 2.52x throughput boost
compiler_args = ['--auto-cast', 'matmult', '--optlevel', '2', '--model-type', 'transformer']

print(f"Tracing for seq_len={SEQ_LEN}...")
print(f"Compiler args: {compiler_args}")
start = time.time()

model_neuron = torch_neuronx.trace(
    model,
    (example_inputs['input_ids'], example_inputs['attention_mask']),
    compiler_args=compiler_args,
)

compile_time = time.time() - start
print(f"Compilation time: {compile_time:.1f}s")

Tracing for seq_len=128...
Compiler args: ['--auto-cast', 'matmult', '--optlevel', '2', '--model-type', 'transformer']


.

.

.

.

.

.

.


Compiler status PASS


Compilation time: 146.0s


## 5. Accuracy Validation

Compare Neuron outputs against CPU reference to verify numerical correctness.

In [5]:
# CPU reference
with torch.no_grad():
    cpu_out = model(example_inputs['input_ids'], example_inputs['attention_mask'])
cpu_hidden = cpu_out[0]  # first element is last_hidden_state

# Neuron inference
neuron_out = model_neuron(example_inputs['input_ids'], example_inputs['attention_mask'])
neuron_hidden = neuron_out[0] if isinstance(neuron_out, tuple) else neuron_out

# Accuracy metrics
cos_sim = F.cosine_similarity(
    cpu_hidden.flatten().unsqueeze(0),
    neuron_hidden.flatten().float().unsqueeze(0)
).item()

max_diff = (cpu_hidden - neuron_hidden.float()).abs().max().item()
mean_diff = (cpu_hidden - neuron_hidden.float()).abs().mean().item()

print(f"Accuracy (Neuron matmult BF16 vs CPU FP32):")
print(f"  Cosine similarity: {cos_sim:.6f}")
print(f"  Max absolute diff: {max_diff:.6f}")
print(f"  Mean absolute diff: {mean_diff:.6f}")
print(f"  Status: {'PASS' if cos_sim > 0.999 else 'FAIL'}")

Accuracy (Neuron matmult BF16 vs CPU FP32):
  Cosine similarity: 1.000009
  Max absolute diff: 0.014795
  Mean absolute diff: 0.000760
  Status: PASS


## 6. Multilingual Inference on Neuron

Run the same multilingual examples through the compiled Neuron model and compare with CPU.

In [6]:
print("Neuron vs CPU embedding comparison:")
print("-" * 60)
for lang, text in texts.items():
    inputs = tokenizer(text, return_tensors="pt", padding="max_length",
                       max_length=SEQ_LEN, truncation=True)
    
    # CPU
    with torch.no_grad():
        cpu_result = model(inputs['input_ids'], inputs['attention_mask'])
    cpu_cls = cpu_result[0][:, 0, :]
    
    # Neuron
    neuron_result = model_neuron(inputs['input_ids'], inputs['attention_mask'])
    neuron_cls = neuron_result[0][:, 0, :] if isinstance(neuron_result, tuple) else neuron_result[:, 0, :]
    
    cos = F.cosine_similarity(
        cpu_cls.flatten().unsqueeze(0),
        neuron_cls.flatten().float().unsqueeze(0)
    ).item()
    print(f"  {lang:>8}: cosine={cos:.6f} {'PASS' if cos > 0.999 else 'FAIL'}")

Neuron vs CPU embedding comparison:
------------------------------------------------------------
   English: cosine=1.000000 PASS


    French: cosine=1.000000 PASS
    German: cosine=1.000000 PASS


   Chinese: cosine=1.000000 PASS
    Arabic: cosine=1.000000 PASS


  Japanese: cosine=1.000000 PASS


## 7. Single-Core Latency

Measure inference latency on a single NeuronCore.

In [7]:
# Warmup
for _ in range(10):
    model_neuron(example_inputs['input_ids'], example_inputs['attention_mask'])

# Measure
latencies = []
for _ in range(100):
    start = time.time()
    model_neuron(example_inputs['input_ids'], example_inputs['attention_mask'])
    latencies.append((time.time() - start) * 1000)

latencies.sort()
p50 = latencies[len(latencies) // 2]
p99 = latencies[int(len(latencies) * 0.99)]
mean_lat = sum(latencies) / len(latencies)

print(f"Single-core latency (seq_len={SEQ_LEN}, BS=1):")
print(f"  Mean: {mean_lat:.2f} ms")
print(f"  P50:  {p50:.2f} ms")
print(f"  P99:  {p99:.2f} ms")
print(f"  Single-core throughput: {1000/mean_lat:.1f} inf/s")

Single-core latency (seq_len=128, BS=1):
  Mean: 4.11 ms
  P50:  4.10 ms
  P99:  4.25 ms
  Single-core throughput: 243.4 inf/s


## 8. DataParallel Scaling (DP=4)

Scale inference across all 4 NeuronCores using `torch_neuronx.DataParallel`.
On trn2.3xlarge with LNC=2, we have 4 logical cores for DP=4.

In [8]:
# DataParallel across all 4 cores
model_dp = torch_neuronx.DataParallel(model_neuron)
num_cores = torch_neuronx.xla_impl.data_parallel.device_count()
print(f"DataParallel on {num_cores} NeuronCores")

# Benchmark at different effective batch sizes
# DataParallel splits the batch across cores, so total_bs must be divisible by num_cores
for total_bs in [4, 16, 32, 64, 128]:
    per_core_bs = total_bs // num_cores
    input_ids = torch.zeros(total_bs, SEQ_LEN, dtype=torch.long)
    attention_mask = torch.ones(total_bs, SEQ_LEN, dtype=torch.long)
    
    # Warmup
    for _ in range(5):
        model_dp(input_ids, attention_mask)
    
    # Measure
    lats = []
    for _ in range(30):
        s = time.time()
        model_dp(input_ids, attention_mask)
        lats.append((time.time() - s) * 1000)
    
    mean_ms = sum(lats) / len(lats)
    throughput = total_bs / (mean_ms / 1000)
    print(f"  BS={total_bs:>3} (per-core={per_core_bs:>2}): {throughput:>8.1f} inf/s  {mean_ms:>7.2f} ms")

print(f"\nNote: Peak throughput requires tracing a model per per-core batch size.")
print(f"This demo uses BS=1 trace with DP splitting, which underestimates peak throughput.")

DataParallel on 4 NeuronCores


  BS=  4 (per-core= 1):    448.3 inf/s     8.92 ms


  BS= 16 (per-core= 4):    466.6 inf/s    34.29 ms


  BS= 32 (per-core= 8):    472.4 inf/s    67.73 ms


  BS= 64 (per-core=16):    474.6 inf/s   134.86 ms


  BS=128 (per-core=32):    476.2 inf/s   268.80 ms

Note: Peak throughput requires tracing a model per per-core batch size.
This demo uses BS=1 trace with DP splitting, which underestimates peak throughput.


## 9. Benchmark Results

Complete benchmark results from dedicated benchmarking (traced per batch size for optimal throughput).

### Neuron Throughput: matmult BF16 Autocast, DP=4

| Seq Length | Batch Size | Per-Core BS | Throughput (inf/s) | Latency P50 (ms) |
|-----------|-----------|-------------|-------------------|------------------|
| 128 | 4 | 1 | 1,029.9 | 3.88 |
| 128 | 16 | 4 | 1,470.3 | 10.87 |
| 128 | 64 | 16 | 1,727.4 | 37.05 |
| **128** | **128** | **32** | **1,763.8** | **72.64** |
| 256 | 4 | 1 | 503.9 | 7.94 |
| 256 | 16 | 4 | 612.1 | 26.14 |
| **256** | **64** | **16** | **687.6** | **93.10** |
| 512 | 4 | 1 | 178.8 | 22.37 |
| 512 | 16 | 4 | 256.5 | 62.38 |
| **512** | **32** | **8** | **276.7** | **115.65** |

### Impact of matmult Autocast

| Seq Length | FP32 Peak (inf/s) | matmult Peak (inf/s) | Speedup | NEFF Size |
|-----------|-------------------|---------------------|---------|-----------|
| 128 | 701.3 | **1,763.8** | **2.52x** | 2,220 MB (vs 2,771 MB) |

The `--auto-cast matmult` compiler flag casts matrix multiplications to BF16 while keeping all other
operations in FP32. This provides a 2.52x throughput improvement with cosine similarity > 0.999999.

## 10. NKI Kernel Exploration

We explored custom NKI (Neuron Kernel Interface) kernels for the attention layers.

### Findings

**nkilib kernels** (attention_cte, mlp) cannot be used via `torch_neuronx.trace()` on SDK 2.29 -- they require the NxDI framework runtime due to Python-level shape/dtype resolution that the trace compiler cannot handle.

**Custom bidirectional flash attention kernel** was written and validated:
- Cosine similarity: 0.999995 (accurate)
- P50 latency: 0.482ms per attention layer (16 heads, seq=128, head_dim=64)
- Estimated 24 layers: 11.57ms vs full model 4.10ms

**Conclusion**: The Neuron compiler's whole-graph optimization (layer fusion, cross-layer pipelining, memory layout optimization) produces **2.8x faster code** than an isolated NKI attention kernel. This is expected -- the compiler can fuse attention + FFN + LayerNorm across all 24 layers simultaneously.

### Key NKI 0.3.0 Learnings

1. `nc_transpose` parameter renamed from `src` to `data`
2. `tensor_reduce` uses `nl.max`/`nl.add` (not `np.max`/`np.add`)
3. `tensor_tensor` has no `divide` op and no broadcasting on CoreV3+ (trn2)
4. Use `nisa.activation(bias=...)` for per-row broadcast operations (subtract max, scale by 1/sum)
5. `PSUM[()] = 0` initialization causes compiler segfault inside sequential_range loops

## 11. GPU Baseline Comparison

Compared against NVIDIA L40S (g6e.xlarge, $1.86/hr) running the same model with PyTorch 2.7.

### Peak Throughput (best batch size per platform)

| Seq Length | Neuron matmult (inf/s) | GPU FP32 (inf/s) | GPU BF16 (inf/s) | Neuron vs FP32 | Neuron vs BF16 |
|-----------|----------------------|-----------------|-----------------|---------------|---------------|
| 128 | **1,763.8** | 384.4 | 1,782.2 | **4.59x** | 0.99x |
| 256 | **687.6** | 179.1 | 877.7 | **3.84x** | 0.78x |
| 512 | **276.7** | 82.1 | 415.2 | **3.37x** | 0.67x |

### Single-Inference Latency (BS=1)

| Seq Length | Neuron (ms) | GPU FP32 (ms) | GPU BF16 (ms) |
|-----------|------------|-------------|-------------|
| 128 | **4.10** | 11.14 | 10.77 |
| 256 | **4.11** | 10.91 | 10.80 |
| 512 | **8.00** | 13.96 | 11.47 |

### Key Takeaways

1. **Neuron dominates FP32 workloads**: 3.4-4.6x faster than GPU FP32. The `matmult` autocast provides free BF16 acceleration without model changes.

2. **GPU BF16 is competitive at throughput**: For small encoder models (559M params), an L40S in BF16 matches Neuron peak throughput at seq128. At longer sequences, GPU BF16 pulls ahead.

3. **Neuron wins on latency**: 2.6-2.7x lower single-inference latency at seq128/256 due to dedicated hardware with near-zero scheduling overhead vs CUDA kernel launch overhead.

4. **Cost context**: trn2 uses capacity block pricing ($2.24/hr) vs GPU on-demand ($1.86/hr). These are different purchasing models and not directly comparable.

## 12. Recommended Configuration

For deploying XLM-RoBERTa-Large on Trainium2:

| Parameter | Recommended Value |
|-----------|------------------|
| Instance | trn2.3xlarge |
| LNC mode | LNC=2 (default) |
| Compiler args | `--auto-cast matmult --optlevel 2 --model-type transformer` |
| Data Parallel | DP=4 (all cores) |
| Optimal batch size | 128 total (32 per core) for seq=128 |
| Expected throughput | ~1,764 inf/s (seq=128) |
| Expected latency | ~4.1 ms (single inference, BS=1) |

### When to use Neuron for encoder models:

- **FP32 workloads**: 3.4-4.6x faster than GPU -- no model changes needed
- **Latency-critical**: 2.6x lower latency than GPU at BS=1
- **Already on Neuron**: Zero additional cost if trn2 capacity is available
- **Mixed workloads**: Run alongside LLM inference on the same trn2 chip

## 13. Save Compiled Model

Save the traced model for reuse without recompilation.

In [9]:
# Save the compiled model
save_path = "xlm_roberta_large_neuron_seq128.pt"
torch.jit.save(model_neuron, save_path)
model_size_mb = os.path.getsize(save_path) / (1024**2)
print(f"Saved compiled model to {save_path} ({model_size_mb:.0f} MB)")
print(f"\nTo reload: model = torch.jit.load('{save_path}')")

Saved compiled model to xlm_roberta_large_neuron_seq128.pt (2117 MB)

To reload: model = torch.jit.load('xlm_roberta_large_neuron_seq128.pt')
